# WiDS Global Datathon 2026 — Wildfire Evacuation-Threat Survival Model

**Final placement: 210 / 1,754 (top 12%)**

---

## The problem

When a wildfire ignites, an incident commander has to decide which communities to
warn, when to warn them, and where to send crews — before anything is certain.
This competition turns that into a **right-censored survival analysis** task.

Using only signals observable in the **first five hours** after the initial
perimeter observation (`t0`), predict the probability that a fire comes within
**5 km of an evacuation zone centroid** by 12h, 24h, 48h, and 72h after `t0 + 5h`.

| | |
|---|---|
| `event = 1` | Fire hit within 72h; `time_to_hit_hours` is the observed hit time |
| `event = 0` | Right-censored; `time_to_hit_hours` is the last observation (≤ 72h) |

## The metric

$$\text{Hybrid} = 0.3 \cdot \text{C-index} + 0.7 \cdot (1 - \text{WeightedBrier})$$

$$\text{WeightedBrier} = 0.3\,B_{24} + 0.4\,B_{48} + 0.3\,B_{72}$$

Brier is **censor-aware**: fires censored *before* a horizon are excluded from
that horizon entirely, because their outcome is genuinely unknown.

The 70/30 split matters for strategy. **Calibration dominates ranking**, so most
of our effort went into probabilities an emergency manager could act on with a
threshold, not just an ordering.

## Our approach in one paragraph

A **Gradient Boosting Survival Analysis** ensemble is the backbone — it handles
censoring natively and produces all four horizons from one internally consistent
survival curve. A second family of **IPCW-weighted LightGBM classifiers**, one per
horizon, corrects its calibration where it drifts (mostly at 48h). The two are
convex-blended per horizon, the 24h column is power-calibrated, the 72h column is
set to a constant, and monotonicity is repaired at the end. With only **221
training rows**, everything is averaged over 40 seeds × 10 configs × 5 folds —
variance reduction was worth more than any single clever model.

## Results

| Metric | OOF |
|---|---|
| **Hybrid score** | **0.97476** |
| C-index | 0.9456 |
| Weighted Brier | 0.01274 |
| Brier @ 12h / 24h / 48h / 72h | 0.05060 / 0.02674 / 0.01180 / 0.00000 |

---

### Notebook layout

1. Environment & configuration
2. Data loading and overview
3. Exploratory analysis
4. Feature engineering
5. Metrics and survival helpers
6. Model A — GBSA ensemble
7. Model B — LightGBM + IPCW
8. What the model learned — feature importance
9. Blending, calibration, and the 72h decision
10. Out-of-fold evaluation
11. Submission
12. Blend-weight search (with an anti-overfitting guard rail)
13. What we learned

---
## 1. Environment & configuration

Every tunable constant lives in this one cell so an experiment can be reproduced
by diffing a single block. Values marked *proven* were chosen on out-of-fold
score and confirmed against the public leaderboard.

In [1]:
import os
import sys
import subprocess
import warnings

warnings.filterwarnings("ignore")


def install(pkg, import_name=None):
    """Install a package only if it is not already importable."""
    try:
        __import__(import_name or pkg)
    except ImportError:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


install("scikit-survival", "sksurv")
install("lightgbm")

import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sksurv.util import Surv
from sksurv.ensemble import GradientBoostingSurvivalAnalysis

# A single consistent look for every figure in the notebook.
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})
INK, HIT, CENSORED = "#2b2b2b", "#d1495b", "#3b7ea1"

print("numpy", np.__version__, "| pandas", pd.__version__, "| lightgbm", lgb.__version__)

Installing scikit-survival ...
numpy 2.2.3 | pandas 2.2.3 | lightgbm 4.5.0


In [2]:
# ─── Paths ───────────────────────────────────────────────────────────────────
DATA_DIR    = "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26"
OUTPUT_PATH = "/kaggle/working/submission.csv"

# ─── Run mode ────────────────────────────────────────────────────────────────
# "fast" ≈ 5 min  (10 seeds)  — use while iterating
# "full" ≈ 40 min (40 seeds)  — use for the submission run
RUN_MODE = "full"
DO_OOF   = True          # OOF roughly doubles runtime but is how we tune honestly
N_FOLDS  = 5

HORIZONS_PRED = [12, 24, 48, 72]

# ─── Blend weights: GBSA vs LightGBM, per horizon (proven on OOF) ────────────
# Short horizons lean almost entirely on the survival model. By 48h the binary
# IPCW classifier carries slightly more of the signal.
W_GBSA_12, W_LGB_12 = 0.97, 0.03
W_GBSA_24, W_LGB_24 = 0.95, 0.05
W_GBSA_48, W_LGB_48 = 0.45, 0.55

# Power calibration on the 24h GBSA column: p -> p ** alpha.
# alpha > 1 shrinks probabilities toward 0, correcting mild over-confidence.
POWER_CAL_24 = 1.1

# ─── 72h handling ────────────────────────────────────────────────────────────
# "constant1" is proven best (public LB 0.97085). Rationale in Section 9.
P72_MODE = "constant1"

# ─── Seeds ───────────────────────────────────────────────────────────────────
# With 221 rows, single-seed OOF swings by several thousandths of a point.
# Heavy seed averaging is the highest-leverage variance reduction available.
GBSA_SEEDS_FULL = (
    123, 456, 789, 777, 666,
    1511, 1523, 2025, 2026, 2033,
    279, 239, 70, 77, 31,
    2024, 2077, 3077, 123456, 654321,
    4640, 841, 7755, 8525, 2701,
    8817, 8864, 4085, 8919, 934,
    4746, 1699, 7401, 7826, 4098,
    2921, 1204, 2752, 8384, 1284,
)
LGB_SEEDS_FULL = (
    123, 456, 789, 777, 666,
    1511, 1523, 2025, 2026, 2033,
    279, 239, 70, 77, 31,
    2024, 2077, 3077, 123456, 654321,
    2034, 2035, 2036, 1984, 1991,
)
GBSA_SEEDS_FAST = LGB_SEEDS_FAST = tuple(range(42, 52))

GBSA_SEEDS = GBSA_SEEDS_FULL if RUN_MODE == "full" else GBSA_SEEDS_FAST
LGB_SEEDS  = LGB_SEEDS_FULL  if RUN_MODE == "full" else LGB_SEEDS_FAST

print(f"mode={RUN_MODE}  GBSA seeds={len(GBSA_SEEDS)}  LGBM seeds={len(LGB_SEEDS)}")

mode=full  GBSA seeds=40  LGBM seeds=25


---
## 2. Data loading and overview

Two facts about this dataset drive every modelling decision that follows:

1. **It is tiny.** 221 training rows, 95 test rows. Any model complex enough to
   memorise it will, and any tuning gain above ~0.001 is almost certainly noise.
2. **It is heavily censored.** 152 of 221 fires (69%) never reach an evacuation
   zone inside the window. Throwing those rows away would discard most of the
   data *and* bias what remains — hence survival models and IPCW.

In [3]:
train_df   = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df    = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

# The survival target, pulled out once and reused throughout the notebook.
event_values = train_df["event"].values
time_values  = train_df["time_to_hit_hours"].values

n_hit = int(train_df["event"].sum())
print(f"train {train_df.shape} | test {test_df.shape}")
print(f"events: {n_hit} hit / {len(train_df) - n_hit} censored "
      f"({n_hit / len(train_df):.1%} event rate)")

train (221, 37) | test (95, 35)
events: 69 hit / 152 censored (31.2% event rate)


---
## 3. Exploratory analysis

Four questions worth answering before writing a model, each of which changed a
decision later in this notebook.

### 3.1 When do fires actually hit?

The observed hit times tell us how much signal each submission horizon carries.
12h is a rare event — only a handful of fires travel far enough that fast — while
by 72h the picture is close to saturated. That imbalance is why the 12h LightGBM
model gets the shallowest trees and the heaviest regularisation (Section 7), and
it is the first hint that the 72h column is a degenerate target.

In [4]:
hits = train_df.loc[train_df["event"] == 1, "time_to_hit_hours"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))

# Left: distribution of observed hit times, with the submission horizons marked.
ax1.hist(hits, bins=24, color=HIT, alpha=0.8, edgecolor="white")
for h in HORIZONS_PRED:
    ax1.axvline(h, color=INK, ls="--", lw=1, alpha=0.5)
    ax1.text(h, ax1.get_ylim()[1] * 0.94, f"{h}h", ha="center",
             fontsize=8, color=INK)
ax1.set_xlabel("hours from t0 + 5h")
ax1.set_ylabel("fires")
ax1.set_title(f"Observed hit times (n = {len(hits)})", loc="left")

# Right: cumulative share of the full training set that has hit by each horizon.
cum = [(hits <= h).sum() / len(train_df) for h in HORIZONS_PRED]
bars = ax2.bar([str(h) for h in HORIZONS_PRED], cum, color=HIT, alpha=0.8)
ax2.bar_label(bars, fmt="%.3f", padding=2, fontsize=8)
ax2.set_ylim(0, max(cum) * 1.25)
ax2.set_xlabel("horizon (h)")
ax2.set_ylabel("share of all fires")
ax2.set_title("Cumulative hit rate by horizon", loc="left")

plt.tight_layout()
plt.show()

for h in HORIZONS_PRED:
    n_by_h = int((hits <= h).sum())
    n_cens = int(((train_df["event"] == 0) &
                  (train_df["time_to_hit_hours"] < h)).sum())
    print(f"  by {h:>2}h: {n_by_h:>3} hits ({n_by_h/len(train_df):5.1%})   "
          f"| censored before horizon (excluded from Brier): {n_cens}")

### 3.2 The censoring pattern — and why the 72h column collapses

The Kaplan–Meier estimator is the right way to look at a censored sample: it uses
the censored fires for as long as they were observed instead of discarding them.

The important detail is not the curve's shape but **where the censored rows sit**.
Censoring in this dataset happens *at* 72h — observation ends with the window — so
there is essentially no fire that survives past 72h while still under observation.
Under the competition's censor-aware Brier rule, every row that is still scored at
72h is therefore a hit. That is the whole basis for the constant-1.0 decision in
Section 9, and it is worth seeing directly rather than taking on faith.

In [5]:
def kaplan_meier(times, events):
    """Kaplan-Meier survival estimate. Returns (time grid, S(t))."""
    order = np.argsort(times)
    t, e = np.asarray(times)[order], np.asarray(events)[order]
    grid, surv, s, n = [0.0], [1.0], 1.0, len(t)
    for ti in np.unique(t):
        at_risk = (t >= ti).sum()
        d = int(((t == ti) & (e == 1)).sum())
        if at_risk > 0 and d > 0:
            s *= 1 - d / at_risk
        grid.append(ti)
        surv.append(s)
    return np.array(grid), np.array(surv)


t_grid, s_grid = kaplan_meier(time_values, event_values)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))

ax1.step(t_grid, s_grid, where="post", color=INK, lw=2)
for h in HORIZONS_PRED:
    s_h = s_grid[np.searchsorted(t_grid, h, side="right") - 1]
    ax1.axvline(h, color=HIT, ls="--", lw=1, alpha=0.6)
    ax1.plot(h, s_h, "o", color=HIT, ms=5)
    ax1.annotate(f"{1 - s_h:.2f}", (h, s_h), textcoords="offset points",
                 xytext=(4, 6), fontsize=8, color=HIT)
ax1.set_xlabel("hours from t0 + 5h")
ax1.set_ylabel("S(t) — not yet threatening")
ax1.set_title("Kaplan-Meier (labels = cumulative hit prob.)", loc="left")
ax1.set_ylim(0, 1.02)

# Where the censored rows end. If they pile up at the window edge, no censored
# fire is ever scored at 72h.
cens_t = time_values[event_values == 0]
ax2.hist(cens_t, bins=24, color=CENSORED, alpha=0.85, edgecolor="white")
ax2.axvline(72, color=HIT, ls="--", lw=1.2)
ax2.set_xlabel("last observed time (h)")
ax2.set_ylabel("censored fires")
ax2.set_title(f"Where censoring happens (n = {len(cens_t)})", loc="left")

plt.tight_layout()
plt.show()

scored_at_72 = ~((event_values == 0) & (time_values < 72))
hits_at_72   = (event_values == 1) & (time_values <= 72)
print(f"rows scored at 72h: {scored_at_72.sum()}  |  of those, hits: "
      f"{hits_at_72[scored_at_72].sum()}")
print(f"-> share of scored rows with label 1 at 72h: "
      f"{hits_at_72[scored_at_72].mean():.3f}")

### 3.3 Proximity and closing speed do most of the work

Two raw signals separate the classes almost on their own: how far the perimeter
already is from the nearest zone, and how fast it is closing. Their **ratio** is a
crude time-to-contact estimate — which is exactly the quantity the target measures,
and exactly why `eta_effective` in Section 4 turned out to be our most useful
engineered feature.

The right-hand panel is the one to check before trusting anything downstream: if
the event rate falls off sharply across distance deciles, a distance-aware model
has real signal to work with. It is also why we kept explicit operational distance
bands (`zone_critical`, `zone_warning`, `zone_safe`) rather than trusting the trees
to rediscover those thresholds from 221 rows.

In [6]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))

# Left: the two dominant raw signals, split by outcome.
for label, mask, colour, name in [
    (1, event_values == 1, HIT, "hit"),
    (0, event_values == 0, CENSORED, "censored"),
]:
    ax1.scatter(train_df.loc[mask, "dist_min_ci_0_5h"] / 1000,
                train_df.loc[mask, "closing_speed_m_per_h"],
                s=26, alpha=0.65, color=colour, label=name,
                edgecolor="white", linewidth=0.4)
ax1.set_xscale("log")
ax1.set_xlabel("distance to nearest zone (km, log)")
ax1.set_ylabel("closing speed (m/h)")
ax1.set_title("Proximity vs closing speed", loc="left")
ax1.legend(frameon=False, fontsize=8)

# Right: event rate by distance decile — the separation is stark.
deciles = pd.qcut(train_df["dist_min_ci_0_5h"], 10, labels=False, duplicates="drop")
rate = train_df.groupby(deciles)["event"].mean()
ax2.bar(rate.index.astype(int) + 1, rate.values, color=HIT, alpha=0.85)
ax2.set_xlabel("distance decile (1 = closest)")
ax2.set_ylabel("event rate")
ax2.set_title("Event rate by distance decile", loc="left")
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.show()

### 3.4 Correlation structure — why we drop columns

Several raw columns measure near-identical things (centroid displacement, centroid
speed, absolute closing speed). On 221 rows, collinear near-duplicates add variance
without adding signal, so `DROP_COLS` in Section 4 removes the redundant ones and
keeps the single clearest representative of each physical quantity.

In [7]:
num = train_df.drop(columns=["event_id"]).select_dtypes("number")
corr = num.corr().fillna(0)

# Rank features by absolute correlation with the outcome, keep the top block.
top = corr["event"].abs().sort_values(ascending=False).head(14).index
sub = corr.loc[top, top]

fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(sub, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(top)), top, rotation=90, fontsize=7)
ax.set_yticks(range(len(top)), top, fontsize=7)
ax.set_title("Correlation, top 14 features by |corr| with event", loc="left")
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.75)
plt.tight_layout()
plt.show()

print("strongest linear associations with the outcome:")
print(corr["event"].drop(["event", "time_to_hit_hours"])
          .abs().sort_values(ascending=False).head(8).round(3).to_string())

---
## 4. Feature engineering

Everything here is computed strictly from the first five hours after `t0`, matching
the competition's information constraint. Nothing touches the target.

The raw columns describe three physical quantities that determine whether a fire
reaches a zone:

| | |
|---|---|
| **Proximity** | how far the perimeter already is from the nearest zone |
| **Kinematics** | how fast it closes that gap — translation *and* radial growth |
| **Intensity** | fuel consumption per hour, and how erratic the perimeter is |

The engineered block mostly expresses interactions between those three, plus an
explicit *time to contact* estimate. That last one, `eta_effective`, is the closest
thing we have to a physical prior for a target measured in hours, and it was the
single most useful feature we added:

$$\text{eta\_effective} = \frac{\text{distance}}{\text{closing speed} + \text{radial growth rate}}$$

A perimeter can close on a zone two ways — the whole fire translates toward it, or
it expands outward in place. Summing both gives a much better ETA than translation
alone, which is why `eta_effective` beats the naive `eta_hours`.

**One deliberate asymmetry:** GBSA is fed the *raw* columns and LightGBM the
*engineered* frame. In ablation, the engineered features measurably hurt GBSA — the
survival trees find those interactions themselves and the extra collinear columns
just add variance — while they clearly helped the shallow per-horizon classifiers.

In [8]:
DROP_COLS = [
    "relative_growth_0_5h", "projected_advance_m", "centroid_displacement_m",
    "centroid_speed_m_per_h", "closing_speed_abs_m_per_h", "area_growth_abs_0_5h",
]

ETA_SENTINEL = 9999.0   # "not closing at all" -> effectively infinite ETA


def create_features(df):
    """Engineered frame for the LightGBM models. No target leakage: every input
    is an observation from the first five hours after t0."""
    r = df.copy()

    dist       = r["dist_min_ci_0_5h"].clip(lower=1)   # metres; avoid /0
    speed      = r["closing_speed_m_per_h"]
    perimeters = r["num_perimeters_0_5h"]
    area_first = r["area_first_ha"]

    # ── Distance transforms ─────────────────────────────────────────────────
    # Risk decays sharply and non-linearly with distance. Hand the trees several
    # monotone re-scalings and let them pick the split that works.
    r["log_distance"]    = np.log1p(dist)
    r["inv_distance"]    = 1 / (dist / 1000 + 0.1)
    r["inv_distance_sq"] = r["inv_distance"] ** 2
    r["sqrt_distance"]   = np.sqrt(dist)
    r["dist_km"]         = dist / 1000
    r["dist_km_sq"]      = (dist / 1000) ** 2
    r["dist_km_cb"]      = (dist / 1000) ** 3
    r["dist_rank"]       = dist.rank(pct=True)

    # ── Area relative to distance ───────────────────────────────────────────
    # A 500 ha fire 2 km away is a different problem from a 5 ha fire 2 km away.
    # Convert area to an equivalent circular radius so both share units.
    fire_radius              = np.sqrt(area_first * 10000 / np.pi)   # ha → m² → m
    r["fire_radius_km"]      = fire_radius / 1000
    r["radius_to_dist"]      = fire_radius / dist
    r["area_to_dist_ratio"]  = area_first / (dist / 1000 + 0.1)
    r["log_area_dist_ratio"] = np.log1p(area_first) - np.log1p(dist)

    # ── Kinematics ──────────────────────────────────────────────────────────
    r["has_movement"] = (perimeters > 1).astype(float)
    closing_pos       = speed.clip(lower=0)
    r["eta_hours"]    = np.where(closing_pos > 0.01,
                                 dist / closing_pos, ETA_SENTINEL).clip(max=ETA_SENTINEL)
    r["log_eta"]      = np.log1p(r["eta_hours"].clip(0, ETA_SENTINEL))

    radial_growth     = r["radial_growth_rate_m_per_h"].clip(lower=0)
    effective_closing = closing_pos + radial_growth          # translation + expansion
    r["effective_closing_speed"] = effective_closing
    r["eta_effective"] = np.where(effective_closing > 0.01,
                                  dist / effective_closing, ETA_SENTINEL).clip(max=ETA_SENTINEL)

    # ── Composite threat scores ─────────────────────────────────────────────
    # `alignment_abs` says whether spread is *pointed at* the zone. Speed only
    # counts when the direction is right.
    r["threat_score"]     = r["alignment_abs"] * speed / np.log1p(dist)
    r["threat_score_sq"]  = r["threat_score"] ** 2
    r["fire_urgency"]     = perimeters * speed
    r["growth_intensity"] = r["area_growth_rate_ha_per_h"] * perimeters

    # ── Operational distance bands (mirrors how commanders bucket proximity) ─
    r["zone_critical"] = (dist < 5000).astype(float)
    r["zone_warning"]  = ((dist >= 5000) & (dist < 10000)).astype(float)
    r["zone_safe"]     = (dist >= 10000).astype(float)

    # ── Temporal context ────────────────────────────────────────────────────
    r["is_summer"]    = r["event_start_month"].isin([6, 7, 8]).astype(float)
    r["is_afternoon"] = ((r["event_start_hour"] >= 12) &
                         (r["event_start_hour"] < 20)).astype(float)

    # ── Cleanup: drop noisy / near-duplicate columns, kill non-finite values ─
    r = r.drop(columns=[c for c in DROP_COLS if c in r.columns])
    return r.replace([np.inf, -np.inf], np.nan).fillna(0)


train_processed = create_features(train_df)
test_processed  = create_features(test_df)

n_feat = len([c for c in train_processed.columns
              if c not in ["event_id", "event", "time_to_hit_hours"]])
print(f"engineered features: {n_feat}")

engineered features: 54


---
## 5. Metrics and survival helpers

We re-implement the leaderboard metric exactly so OOF numbers are directly
comparable to the public score. Two subtleties are easy to get wrong:

**Censor-aware Brier.** A fire censored *before* horizon `H` has an unknown
outcome at `H` and must be dropped from that horizon's score — not scored as a 0.
Getting this wrong makes local validation look better than the leaderboard.

**IPCW.** Simply dropping censored rows biases the binary classifiers: fires that
leave observation early are not a random sample. Inverse-probability-of-censoring
weighting corrects for it by up-weighting each retained row by $1/G(t)$, where $G$
is the Kaplan–Meier estimate of the *censoring* survival function.

In [9]:
def compute_c_index(time, event, risk):
    """Harrell's concordance index.

    A pair (i, j) is comparable when i experienced the event and hit strictly
    earlier than j was last observed. Concordant when i got the higher risk;
    ties score half credit. Vectorised over the full pair matrix.
    """
    time, event, risk = map(np.asarray, (time, event, risk))
    comparable = (event[:, None] == 1) & (time[:, None] < time[None, :])
    n_comparable = comparable.sum()
    if n_comparable == 0:
        return 0.5
    higher = (risk[:, None] >  risk[None, :]) & comparable
    tied   = (risk[:, None] == risk[None, :]) & comparable
    return float((higher.sum() + 0.5 * tied.sum()) / n_comparable)


def compute_brier(time, event, prob, horizon):
    """Censor-aware Brier score at `horizon` (lower is better)."""
    valid = ~((event == 0) & (time < horizon))       # unknown outcome → excluded
    if valid.sum() == 0:
        return 0.25
    y_true = ((event == 1) & (time <= horizon)).astype(float)[valid]
    return float(np.mean((np.clip(prob[valid], 0, 1) - y_true) ** 2))


def compute_hybrid_score(time, event, p24, p48, p72):
    """The leaderboard metric: 0.3·C-index + 0.7·(1 − WeightedBrier).

    The risk score fed to the C-index reuses the Brier horizon weights, so
    ranking and calibration are optimised jointly rather than pulling apart.
    """
    risk  = 0.3 * p24 + 0.4 * p48 + 0.3 * p72
    c_idx = compute_c_index(time, event, risk)
    b24   = compute_brier(time, event, p24, 24)
    b48   = compute_brier(time, event, p48, 48)
    b72   = compute_brier(time, event, p72, 72)
    wbrier = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    return 0.3 * c_idx + 0.7 * (1 - wbrier), c_idx, wbrier


def make_binary_target(time_vals, event_vals, horizon):
    """Survival target → binary target at `horizon`.

    Returns (y, mask). `mask` is False for rows censored before the horizon,
    whose label is genuinely unknowable; they are dropped from training and
    from evaluation alike.
    """
    unknown = (event_vals == 0) & (time_vals < horizon)
    y = ((event_vals == 1) & (time_vals <= horizon)).astype(float)
    return y, ~unknown


def compute_ipcw_weights(times, events, horizon):
    """Inverse-probability-of-censoring weights, 1 / G(t).

    G is floored at 0.01 so weights cannot explode in the tail.
    """
    unique_t = np.sort(np.unique(times))
    surv = np.ones(len(unique_t))
    for i, t in enumerate(unique_t):
        at_risk       = (times >= t).sum()
        censored_at_t = ((times == t) & (events == 0)).sum()
        if at_risk > 0:
            surv[i] = 1 - censored_at_t / at_risk
        if i > 0:
            surv[i] *= surv[i - 1]

    def G(t):
        idx = np.searchsorted(unique_t, t, side="right") - 1
        return max(surv[idx], 0.01) if idx >= 0 else 1.0

    weights = np.ones(len(times))
    for i in range(len(times)):
        if events[i] == 1 and times[i] <= horizon:
            weights[i] = 1.0 / G(times[i])
        elif times[i] >= horizon:
            weights[i] = 1.0 / G(horizon)
    return weights


def enforce_monotonicity(preds):
    """Clip to [0, 1] and enforce p12 ≤ p24 ≤ p48 ≤ p72 row-wise.

    A submission requirement, and also just correct: a cumulative hit
    probability cannot fall as the horizon extends. Blending two models per
    column can break it, so we repair with a running maximum.
    """
    result = np.clip(preds, 0, 1)
    for i in range(1, result.shape[1]):
        result[:, i] = np.maximum(result[:, i], result[:, i - 1])
    return result


def get_surv_predictions(model, X):
    """Evaluate a fitted survival model at the four competition horizons.

    scikit-survival returns step functions defined only on the observed time
    grid, so horizons are clipped into each function's domain first.
    Returns hit probabilities 1 − S(t), shape (n, 4).
    """
    surv_fns = model.predict_survival_function(X)
    preds = np.empty((len(surv_fns), len(HORIZONS_PRED)), dtype=float)
    for i, fn in enumerate(surv_fns):
        t_min, t_max = fn.domain
        preds[i, :] = fn(np.clip(HORIZONS_PRED, t_min, t_max))
    return 1.0 - preds

---
## 6. Model A — Gradient Boosting Survival Analysis (backbone)

GBSA is the natural fit for this target. It handles right-censoring natively, and
one fit yields the entire survival curve, so all four horizons come out
internally consistent instead of needing four separate models stitched together.

**Ten deliberately diverse configurations.** Depth stays at 2–4 and learning rates
stay low because the training set is tiny. Crucially, diversity *across* configs
did more for stability than tuning any single config well — the ensemble average
is what we are actually optimising.

**Folds are stratified on the event indicator.** With only 69 events, an unlucky
random split can leave a fold with a badly skewed hit ratio.

Total fits in full mode: 10 configs × 40 seeds × 5 folds = **2,000 models**.

In [10]:
# GBSA gets the RAW columns — see the note in Section 4.
X_surv_train = train_df.drop(columns=["event_id", "event", "time_to_hit_hours"])
X_surv_test  = test_df.drop(columns=["event_id"])

y_surv = Surv.from_arrays(
    event=train_df["event"].astype(bool),
    time=train_df["time_to_hit_hours"],
)

GBSA_CONFIGS = [
    {"learning_rate": 0.010, "subsample": 0.70, "max_depth": 3, "min_samples_leaf": 12, "min_samples_split": 3, "n_estimators": 1200},
    {"learning_rate": 0.010, "subsample": 0.85, "max_depth": 3, "min_samples_leaf": 15, "min_samples_split": 3, "n_estimators": 1200},
    {"learning_rate": 0.010, "subsample": 0.60, "max_depth": 3, "min_samples_leaf": 12, "min_samples_split": 3, "n_estimators": 1200},
    {"learning_rate": 0.005, "subsample": 0.85, "max_depth": 3, "min_samples_leaf": 12, "min_samples_split": 3, "n_estimators": 2000},
    {"learning_rate": 0.010, "subsample": 0.85, "max_depth": 3, "min_samples_leaf": 20, "min_samples_split": 3, "n_estimators": 1400},
    {"learning_rate": 0.008, "subsample": 0.75, "max_depth": 2, "min_samples_leaf": 15, "min_samples_split": 4, "n_estimators": 1500},
    {"learning_rate": 0.015, "subsample": 0.70, "max_depth": 3, "min_samples_leaf": 10, "min_samples_split": 3, "n_estimators": 1000},
    {"learning_rate": 0.005, "subsample": 0.90, "max_depth": 3, "min_samples_leaf": 18, "min_samples_split": 5, "n_estimators": 2500},
    {"learning_rate": 0.010, "subsample": 0.80, "max_depth": 4, "min_samples_leaf": 12, "min_samples_split": 3, "n_estimators": 1200},
    {"learning_rate": 0.020, "subsample": 0.65, "max_depth": 3, "min_samples_leaf": 10, "min_samples_split": 3, "n_estimators": 800},
]

print(f"GBSA features: {X_surv_train.shape[1]}")
print(f"total fits: {len(GBSA_CONFIGS)} × {len(GBSA_SEEDS)} × {N_FOLDS} = "
      f"{len(GBSA_CONFIGS) * len(GBSA_SEEDS) * N_FOLDS}")

GBSA features: 34
total fits: 10 × 40 × 5 = 2000


In [11]:
n_h       = len(HORIZONS_PRED)
oof_gbsa  = np.zeros((len(X_surv_train), n_h)) if DO_OOF else None
test_gbsa = np.zeros((len(X_surv_test),  n_h))

print(f"[GBSA] {len(GBSA_CONFIGS)} configs × {len(GBSA_SEEDS)} seeds × {N_FOLDS} folds")

for cfg_idx, cfg in enumerate(GBSA_CONFIGS, 1):
    cfg_oof  = np.zeros((len(X_surv_train), n_h)) if DO_OOF else None
    cfg_test = np.zeros((len(X_surv_test),  n_h))

    for seed in GBSA_SEEDS:
        seed_oof  = np.zeros((len(X_surv_train), n_h)) if DO_OOF else None
        seed_test = np.zeros((len(X_surv_test),  n_h))
        cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

        for tr_idx, va_idx in cv.split(X_surv_train, event_values):
            model = GradientBoostingSurvivalAnalysis(**{**cfg, "random_state": seed})
            model.fit(X_surv_train.iloc[tr_idx], y_surv[tr_idx])
            if DO_OOF:
                seed_oof[va_idx] = get_surv_predictions(model, X_surv_train.iloc[va_idx])
            seed_test += get_surv_predictions(model, X_surv_test) / N_FOLDS

        if DO_OOF:
            cfg_oof += seed_oof / len(GBSA_SEEDS)
        cfg_test += seed_test / len(GBSA_SEEDS)

    if DO_OOF:
        oof_gbsa += cfg_oof / len(GBSA_CONFIGS)
    test_gbsa += cfg_test / len(GBSA_CONFIGS)
    print(f"  config {cfg_idx}/{len(GBSA_CONFIGS)} done")

# Keep pristine copies: calibration is applied later and the grid search in
# Section 12 needs to start from uncalibrated predictions every iteration.
oof_gbsa_raw  = oof_gbsa.copy() if DO_OOF else None
test_gbsa_raw = test_gbsa.copy()

[GBSA] 10 configs × 40 seeds × 5 folds
  config 1/10 done
  config 2/10 done
  config 3/10 done
  config 4/10 done
  config 5/10 done
  config 6/10 done
  config 7/10 done
  config 8/10 done
  config 9/10 done
  config 10/10 done


---
## 7. Model B — LightGBM + IPCW (calibration correction)

One binary classifier per horizon (12h / 24h / 48h), trained only on rows whose
outcome at that horizon is *known*, with IPCW weights to undo the resulting
selection bias.

Why bother when GBSA already gives us everything? Because GBSA carries a
proportional-hazards-style structure that a direct classifier does not, and at 48h
that structure costs calibration. The classifier optimises exactly the quantity
the Brier score measures, at exactly the horizon it is measured — and 48h is the
heaviest-weighted term in the metric.

Per-horizon hyperparameters differ for a reason: **12h is the sparsest target**
(few fires travel that fast), so it gets the shallowest trees and the most
aggressive regularisation.

One bookkeeping note: OOF predictions for rows excluded at a horizon are filled in
from the last fold model purely to keep the array aligned for blending. The
censor-aware Brier masks them out again, so they never touch a reported score.

In [12]:
# LightGBM gets the ENGINEERED frame.
X_lgb_train = train_processed.drop(columns=["event_id", "event", "time_to_hit_hours"])
X_lgb_test  = test_processed.drop(columns=["event_id"])

LGB_CONFIGS = {
    12: {"max_depth": 2, "learning_rate": 0.03, "n_estimators": 200,       # sparsest
         "subsample": 0.7, "colsample_bytree": 0.7, "min_child_samples": 10,
         "reg_alpha": 1.0, "reg_lambda": 3.0, "num_leaves": 4},
    24: {"max_depth": 3, "learning_rate": 0.03, "n_estimators": 300,
         "subsample": 0.7, "colsample_bytree": 0.7, "min_child_samples": 8,
         "reg_alpha": 0.5, "reg_lambda": 2.0, "num_leaves": 7},
    48: {"max_depth": 2, "learning_rate": 0.05, "n_estimators": 200,
         "subsample": 0.8, "colsample_bytree": 0.8, "min_child_samples": 5,
         "reg_alpha": 0.1, "reg_lambda": 1.0, "num_leaves": 4},
}

lgb_oof, lgb_test = {}, {}
lgb_importance = {}          # gain importance, averaged over seeds (Section 8)
print(f"[LGBM] {len(LGB_SEEDS)} seeds × 3 horizons (12h, 24h, 48h)")

for horizon in [12, 24, 48]:
    y_bin, mask  = make_binary_target(time_values, event_values, horizon)
    valid_idx    = np.where(mask)[0]
    censored_idx = np.where(~mask)[0]
    cfg          = LGB_CONFIGS[horizon]

    all_oof  = np.zeros(len(X_lgb_train))
    all_test = np.zeros(len(X_lgb_test))
    all_imp  = np.zeros(X_lgb_train.shape[1])

    for seed in LGB_SEEDS:
        seed_oof, last_model = np.zeros(len(X_lgb_train)), None
        cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

        for tr_v, va_v in cv.split(valid_idx, y_bin[mask]):
            tr_idx, va_idx = valid_idx[tr_v], valid_idx[va_v]
            weights = compute_ipcw_weights(time_values[tr_idx],
                                           event_values[tr_idx], horizon)
            model = lgb.LGBMClassifier(**cfg, objective="binary",
                                       random_state=seed, verbose=-1)
            model.fit(X_lgb_train.iloc[tr_idx], y_bin[tr_idx], sample_weight=weights)
            seed_oof[va_idx] = model.predict_proba(X_lgb_train.iloc[va_idx])[:, 1]
            last_model = model

        # Alignment fill for masked rows — never scored, see the note above.
        if len(censored_idx) and last_model is not None:
            seed_oof[censored_idx] = last_model.predict_proba(
                X_lgb_train.iloc[censored_idx])[:, 1]
        all_oof += seed_oof

        # Refit on all valid rows for the test prediction.
        weights_full = compute_ipcw_weights(time_values[valid_idx],
                                            event_values[valid_idx], horizon)
        model_full = lgb.LGBMClassifier(**cfg, objective="binary",
                                        random_state=seed, verbose=-1)
        model_full.fit(X_lgb_train.iloc[valid_idx], y_bin[valid_idx],
                       sample_weight=weights_full)
        all_test += model_full.predict_proba(X_lgb_test)[:, 1]
        all_imp  += model_full.booster_.feature_importance(importance_type="gain")

    lgb_oof[horizon]  = all_oof  / len(LGB_SEEDS)
    lgb_test[horizon] = all_test / len(LGB_SEEDS)
    lgb_importance[horizon] = pd.Series(all_imp / len(LGB_SEEDS),
                                        index=X_lgb_train.columns)

    b = compute_brier(time_values, event_values, np.clip(lgb_oof[horizon], 0, 1), horizon)
    print(f"  LGBM {horizon:>2}h  Brier={b:.5f}")

[LGBM] 25 seeds × 3 horizons (12h, 24h, 48h)
  LGBM 12h  Brier=0.05624
  LGBM 24h  Brier=0.02594
  LGBM 48h  Brier=0.01272


---
## 8. What the model learned — feature importance

Gain importance from the LightGBM models, averaged over all seeds. Two things are
worth reading off this rather than just admiring it.

**The physics holds up.** Distance and the ETA terms dominate at every horizon,
which is what a fire behaviour analyst would predict and a useful sanity check
that nothing has leaked or gone sideways.

**The ranking shifts with the horizon.** The printout below reports exactly which
features gain and lose ground between 12h and 48h — worth reading, because the
shift is the empirical justification for training a separate model per horizon
rather than one model with a horizon feature. The hypothesis going in was that
short horizons are dominated by raw proximity (only fires already close can arrive
that fast) while longer ones give growth and directional terms room to matter.

In [13]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.6), sharex=False)

for ax, horizon in zip(axes, [12, 24, 48]):
    top = lgb_importance[horizon].sort_values(ascending=False).head(12)[::-1]
    share = top / lgb_importance[horizon].sum()
    ax.barh(range(len(top)), share.values, color=HIT, alpha=0.85)
    ax.set_yticks(range(len(top)), top.index, fontsize=8)
    ax.set_xlabel("share of total gain")
    ax.set_title(f"{horizon}h model", loc="left")

plt.suptitle("LightGBM gain importance, top 12 per horizon",
             x=0.01, ha="left", fontsize=11)
plt.tight_layout()
plt.show()

# Which features move up or down as the horizon extends?
shares = pd.DataFrame({h: lgb_importance[h] / lgb_importance[h].sum()
                       for h in [12, 24, 48]})
shift = (shares[48] - shares[12]).sort_values()
print("gained importance from 12h → 48h:")
print(shift.tail(5).round(4).to_string())
print("\nlost importance from 12h → 48h:")
print(shift.head(5).round(4).to_string())

---
## 9. Blending, calibration, and the 72h decision

Four steps, in order:

**1. Power-calibrate the 24h column.** $p \mapsto p^{1.1}$. An exponent above 1
shrinks probabilities toward zero, correcting GBSA's mild over-confidence at 24h.
It is a one-parameter monotone transform, so it cannot change the ranking — only
the calibration term.

**2. Convex-blend per horizon.** The weights shift with the horizon, and the shift
is informative: at 12h and 24h the survival model is near-unbeatable (0.97 / 0.95),
but by 48h the IPCW classifier earns the majority (0.55). That is the horizon where
modelling the Brier target directly pays off most.

**3. Set the 72h column to a constant 1.0.** This looks like a hack and is worth
justifying properly. Under the competition's censor-aware rule, Brier@72h scores
only two kinds of row — fires that hit by 72h (label 1) and fires censored *after*
72h (label 0). In this dataset the second group is empty: censoring happens at 72h,
not beyond it, so every row that survives the mask is a hit. Predicting 1.0 gives
**Brier@72h = 0.0 exactly**, and since 72h carries 0.3 of the weighted Brier, that
is worth roughly 0.021 of hybrid score outright. It also cannot hurt the C-index: a
constant column adds the same amount to every fire's risk score, leaving the
ranking untouched.

The honest caveat — this is fitted to a quirk of *this* dataset's censoring
mechanism, not to wildfire physics. An operational system would emit a real 72h
probability. We flag it clearly rather than dress it up as insight.

**4. Repair monotonicity.** Blending different models per column can produce
$p_{48} < p_{24}$. A running row-wise maximum fixes it.

In [14]:
def blend(gbsa, lgb_preds,
          w12=W_GBSA_12, w24=W_GBSA_24, w48=W_GBSA_48,
          power_cal_24=POWER_CAL_24, p72_mode=P72_MODE):
    """Combine the survival backbone with the per-horizon classifiers.

    Does not mutate `gbsa` — the grid search relies on that.
    """
    out = gbsa.copy()
    if power_cal_24 != 1.0:                       # step 1
        out[:, 1] = np.clip(out[:, 1] ** power_cal_24, 0, 1)

    blended = out.copy()                          # step 2
    blended[:, 0] = w12 * out[:, 0] + (1 - w12) * lgb_preds[12]
    blended[:, 1] = w24 * out[:, 1] + (1 - w24) * lgb_preds[24]
    blended[:, 2] = w48 * out[:, 2] + (1 - w48) * lgb_preds[48]

    if p72_mode == "constant1":                   # step 3
        blended[:, 3] = 1.0

    return enforce_monotonicity(blended)          # step 4

---
## 10. Out-of-fold evaluation

The number to watch is the hybrid score. `OOF ≈ 0.9748` corresponded to roughly
`0.970` on the public leaderboard — a stable gap we saw hold across submissions,
which is what let us trust OOF for decisions instead of burning submissions.

In [15]:
if DO_OOF:
    oof_final = blend(oof_gbsa_raw, lgb_oof)

    hybrid, c_idx, wbrier = compute_hybrid_score(
        time_values, event_values,
        oof_final[:, 1], oof_final[:, 2], oof_final[:, 3])

    briers = {h: compute_brier(time_values, event_values, oof_final[:, i], h)
              for i, h in enumerate(HORIZONS_PRED)}

    print("=" * 62)
    print(f"OOF hybrid  {hybrid:.5f}   C-index {c_idx:.4f}   WBrier {wbrier:.5f}")
    print("  " + "  ".join(f"B{h}={briers[h]:.5f}" for h in HORIZONS_PRED))
    print("=" * 62)

OOF hybrid  0.97476   C-index 0.9456   WBrier 0.01274
  B12=0.05060  B24=0.02674  B48=0.01180  B72=0.00000


---
## 11. Submission

Validated against every rule the competition checker enforces — schema, exact ID
match, `[0, 1]` range, and row-wise monotonicity — before writing to disk. Cheap
insurance against losing a submission slot to a formatting slip.

In [16]:
def validate_submission(sub, sample):
    """Raise if the submission would be rejected by the competition validator."""
    required = ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]
    assert list(sub.columns) == required,          f"bad schema: {list(sub.columns)}"
    assert len(sub) == len(sample),                "row count differs from sample"
    assert sub["event_id"].is_unique,              "duplicate event_id"
    assert set(sub["event_id"]) == set(sample["event_id"]), "event_id mismatch"
    assert sub[required[1:]].notna().all().all(),  "NaN probability"

    probs = sub[required[1:]].to_numpy()
    assert probs.min() >= 0 and probs.max() <= 1,  "probability outside [0, 1]"
    assert (np.diff(probs, axis=1) >= -1e-12).all(), "monotonicity violated"


test_final = blend(test_gbsa_raw, lgb_test)

submission = pd.DataFrame({
    "event_id": test_df["event_id"].values,
    "prob_12h": test_final[:, 0],
    "prob_24h": test_final[:, 1],
    "prob_48h": test_final[:, 2],
    "prob_72h": test_final[:, 3],
})
submission = sample_sub[["event_id"]].merge(submission, on="event_id", how="left")

validate_submission(submission, sample_sub)
submission.to_csv(OUTPUT_PATH, index=False)

print(f"saved → {OUTPUT_PATH}")
print(submission.head())

saved → /kaggle/working/submission.csv
   event_id  prob_12h  prob_24h  prob_48h  prob_72h
0  10662602  0.014767  0.025757  0.025757       1.0
1  13353600  0.710987  0.947032  0.988383       1.0
2  13942327  0.014638  0.025467  0.025467       1.0
3  16112781  0.668651  0.930789  0.983814       1.0
4  17132808  0.019044  0.033587  0.033587       1.0


---
## 12. Blend-weight search — with a guard rail

A small grid over the two blend weights and the calibration exponent. The guard
rail is the point of this section, not the search.

On 221 rows, **a large apparent gain is a warning sign, not a win.** If re-tuning
three parameters moves OOF by more than about 0.001, the search has almost
certainly fit fold noise rather than signal, and the leaderboard will not follow.
We only adopt small, stable gains. This rule cost us a few tenths of a point on
paper several times, and saved us from at least two shakeup candidates.

In [17]:
if DO_OOF:
    TRUST_THRESHOLD = 0.001
    best_score, best_params = -np.inf, None

    for w24 in [0.90, 0.93, 0.95, 0.97]:
        for w48 in [0.40, 0.43, 0.45, 0.48]:
            for pcal in [1.0, 1.05, 1.10, 1.15]:
                arr = blend(oof_gbsa_raw, lgb_oof, w24=w24, w48=w48, power_cal_24=pcal)
                score, _, _ = compute_hybrid_score(
                    time_values, event_values, arr[:, 1], arr[:, 2], arr[:, 3])
                if score > best_score:
                    best_score = score
                    best_params = {"W_GBSA_24": w24, "W_GBSA_48": w48,
                                   "POWER_CAL_24": pcal}

    gain = best_score - hybrid
    print(f"best OOF {best_score:.5f}  vs baseline {hybrid:.5f}  (gain {gain:+.5f})")
    print(f"best params: {best_params}")

    if gain > TRUST_THRESHOLD:
        print(f"gain > {TRUST_THRESHOLD} → likely fitting fold noise. NOT adopted.")
    elif gain > 0:
        print(f"gain < {TRUST_THRESHOLD} → small and stable. Safe to adopt.")
    else:
        print("no improvement; keeping current weights.")

best OOF 0.97491  vs baseline 0.97476  (gain +0.00015)
best params: {'W_GBSA_24': 0.9, 'W_GBSA_48': 0.4, 'POWER_CAL_24': 1.15}
gain < 0.001 → small and stable. Safe to adopt.


---
## 13. What we learned

**Variance reduction beat model selection.** On 221 rows, going from 5 seeds to 40
bought more stable leaderboard score than any architecture change we tried. Most of
our compute went into averaging, not searching.

**Read the metric before modelling.** The 70/30 calibration/ranking split, the
0.4 weight on 48h, and the censor-aware exclusion rule each changed a concrete
decision — respectively: prioritising Brier over C-index, giving 48h its own
dedicated classifier, and the 72h constant.

**Two model families with different failure modes beat one tuned family.** GBSA and
IPCW-LightGBM disagree most at 48h, which is exactly where the blend helps most. A
third family of similar boosted trees added nothing.

**A gain you cannot explain is a gain you should not take.** The 0.001 trust
threshold in Section 12 was the most valuable line of code we wrote, and it is the
one that produces no output at all.

### What we would do next

- **Conformal prediction intervals** on each horizon. Emergency managers acting on a
  threshold need to know when the model is uncertain, not just what it predicts.
- **A real 72h model**, so the pipeline generalises past this dataset's censoring quirk.
- **Spatial cross-validation** by region, to check the model is not leaning on
  geography-specific patterns that would not transfer to a new fire season.
- **Cost-sensitive thresholds**, since a missed evacuation and a false alarm are not
  remotely symmetric costs.

---

*WiDS Global Datathon 2026 · organised in partnership with Watch Duty.*